In [2]:
from pathlib import Path
import re
import logging

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

logging.basicConfig(level=logging.WARNING)

# ── Paths ─────────────────────────────────────────────────────────────────────
DOCS_DIR   = Path("documents")
OUTPUT_DIR = Path("output")

# All PDFs to process — edit this list or use glob to pick a subset
PDF_FILES = sorted(DOCS_DIR.glob("*.pdf"))

print(f"Found {len(PDF_FILES)} PDF(s):")
for p in PDF_FILES:
    print(f"  {p.name}")

Found 3 PDF(s):
  Session 06 - Regular Expressions.pdf
  Session 07 - Information Retrieval.pdf
  Session 09 - Naive Bayes.pdf


In [3]:
# ── Docling converter setup ────────────────────────────────────────────────────
#
# generate_picture_images=True   → populates pic.image.pil_image for every figure
# do_formula_enrichment=True     → runs CodeFormulaV2 (downloads ~1 GB on first run)
#                                  to convert detected formula regions to LaTeX
#                                  set False to skip formula parsing
# images_scale=2.0               → render pages at 2x resolution for sharper crops

FORMULA_ENRICHMENT = True   # set False if CodeFormulaV2 model is not available

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0
pipeline_options.do_formula_enrichment = FORMULA_ENRICHMENT

converter = DocumentConverter(
    format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
)

print("Converter ready.")
print(f"  formula enrichment: {FORMULA_ENRICHMENT}")

Converter ready.
  formula enrichment: True


In [4]:
# ── Helper: save pictures to disk ─────────────────────────────────────────────

def save_pictures(doc, img_dir: Path) -> list:
    """
    Save every PictureItem image from a DoclingDocument to img_dir.
    Returns an ordered list of saved Path objects (None for pictures with no data).
    """
    img_dir.mkdir(parents=True, exist_ok=True)
    paths = []

    for i, pic in enumerate(doc.pictures):
        if pic.image is None:
            paths.append(None)
            continue

        pil_img = pic.image.pil_image
        # Skip tiny images (decorative lines / separators)
        if pil_img.width < 40 or pil_img.height < 40:
            paths.append(None)
            continue

        path = img_dir / f"img_{i:03d}.png"
        pil_img.save(path, format="PNG")
        paths.append(path)

    saved = sum(1 for p in paths if p is not None)
    print(f"    Saved {saved}/{len(paths)} images -> {img_dir}")
    return paths


# ── Helper: build markdown with real image references ─────────────────────────

def build_markdown(doc, img_paths: list, md_path: Path) -> str:
    """
    Export docling document to markdown, replacing every <!-- image --> placeholder
    with a proper ![fig_N](relative/path.png) reference.

    When do_formula_enrichment=True, docling automatically exports formula regions
    as LaTeX (display: $$...$$, inline: $...$) via CodeFormulaV2.

    md_path is used only to compute relative image paths.
    """
    PLACEHOLDER = "<!-- image -->"

    raw_md = doc.export_to_markdown(
        image_mode="placeholder",
        image_placeholder=PLACEHOLDER,
        page_break_placeholder="\n\n---\n\n",
        escape_html=False,        # keep & < > readable
        escape_underscores=False, # avoid \_word\_ noise in headings
    )

    placeholder_count = raw_md.count(PLACEHOLDER)
    if placeholder_count != len(img_paths):
        print(
            f"    [WARN] placeholder count ({placeholder_count}) != "
            f"picture count ({len(img_paths)}) -- some images may be misaligned"
        )

    parts = raw_md.split(PLACEHOLDER)
    result = []
    img_idx = 0

    for i, part in enumerate(parts):
        result.append(part)
        if i < len(parts) - 1:  # not the last segment
            if img_idx < len(img_paths) and img_paths[img_idx] is not None:
                rel = img_paths[img_idx].relative_to(md_path.parent)
                result.append(f"![fig_{img_idx}]({rel})")
            else:
                result.append(PLACEHOLDER)  # no image saved — keep placeholder
            img_idx += 1

    return "".join(result)


print("Helpers defined.")

Helpers defined.


In [5]:
# ── Main pipeline ──────────────────────────────────────────────────────────────

def process_pdf(pdf_path: Path, output_dir: Path, converter) -> Path:
    """
    Convert a single PDF to a self-contained markdown file.

    Output layout:
        output_dir/<stem>/
            <stem>.md          <- markdown with image references and LaTeX formulas
            images/
                img_000.png
                img_001.png
                ...
    """
    stem = pdf_path.stem
    doc_dir = output_dir / stem
    img_dir = doc_dir / "images"
    md_path = doc_dir / f"{stem}.md"

    doc_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{stem}] Converting...")
    result = converter.convert(pdf_path)
    doc = result.document
    print(f"    {len(doc.pictures)} pictures, {len(doc.tables)} tables")

    # Save images
    img_paths = save_pictures(doc, img_dir)

    # Build and save markdown
    md = build_markdown(doc, img_paths, md_path)
    md_path.write_text(md, encoding="utf-8")

    print(f"    Written: {md_path}  ({len(md):,} chars)")
    return md_path


print("Pipeline function defined.")

Pipeline function defined.


In [6]:
# ── Run on all PDFs ────────────────────────────────────────────────────────────

output_paths = []
for pdf_path in PDF_FILES:
    out = process_pdf(pdf_path, output_dir=OUTPUT_DIR / "markdown", converter=converter)
    output_paths.append(out)

print(f"\nDone. {len(output_paths)} file(s) written.")
for p in output_paths:
    print(f"  {p}")

[Session 06 - Regular Expressions] Converting...


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 22274.66it/s]
The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


    23 pictures, 4 tables
    Saved 23/23 images -> output/markdown/Session 06 - Regular Expressions/images
    Written: output/markdown/Session 06 - Regular Expressions/Session 06 - Regular Expressions.md  (15,254 chars)
[Session 07 - Information Retrieval] Converting...


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


    30 pictures, 0 tables
    Saved 30/30 images -> output/markdown/Session 07 - Information Retrieval/images
    Written: output/markdown/Session 07 - Information Retrieval/Session 07 - Information Retrieval.md  (11,194 chars)
[Session 09 - Naive Bayes] Converting...
    34 pictures, 1 tables
    Saved 34/34 images -> output/markdown/Session 09 - Naive Bayes/images
    Written: output/markdown/Session 09 - Naive Bayes/Session 09 - Naive Bayes.md  (10,921 chars)

Done. 3 file(s) written.
  output/markdown/Session 06 - Regular Expressions/Session 06 - Regular Expressions.md
  output/markdown/Session 07 - Information Retrieval/Session 07 - Information Retrieval.md
  output/markdown/Session 09 - Naive Bayes/Session 09 - Naive Bayes.md


In [7]:
# ── Quick preview of one output ───────────────────────────────────────────────

if output_paths:
    md_text = output_paths[0].read_text(encoding="utf-8")
    print(f"Preview: {output_paths[0].name}  ({len(md_text):,} chars)")
    print("-" * 60)
    print(md_text[:3000])
    print("-" * 60)
    # Report image refs and formula blocks found
    img_refs = re.findall(r'!\[fig_\d+\]\([^)]+\)', md_text)
    formula_blocks = re.findall(r'\$\$[^$]+\$\$', md_text, re.DOTALL)
    inline_formulas = re.findall(r'(?<!\$)\$(?!\$)[^$\n]+\$(?!\$)', md_text)
    print(f"\nImage references : {len(img_refs)}")
    print(f"Display formulas : {len(formula_blocks)}")
    print(f"Inline formulas  : {len(inline_formulas)}")
    if formula_blocks:
        print("\nFirst display formula:")
        print(formula_blocks[0][:300])

Preview: Session 06 - Regular Expressions.md  (15,254 chars)
------------------------------------------------------------
![fig_0](images/img_000.png)

![fig_1](images/img_001.png)

## AI: NATURAL LANGUAGE PROCESSING & SEMANTIC ANALYSIS

BCSAI JAN-2025

PROF. DR. JUANJO MANJARÍN



---



![fig_2](images/img_002.png)

## SESSION 6: REGULAR EXPRESSIONS

Vector Space Models and Semantic Representation

![fig_3](images/img_003.png)



---



![fig_4](images/img_004.png)

## SESSION CONTENTS

- From Symbols to Spaces
- The Bag-of-Words Representation
- The Term-Document Matrix
- Words, Vectors, and Distributional Meaning
- Geometric View of Meaning
- Term Frequency-Inverse Document Frequency
- Latent Semantic Analysis
- HAL: Contextual Co-occurrence
- Practice

![fig_5](images/img_005.png)



---



## COURSE PROJECT PIPELINE

![fig_6](images/img_006.png)

![fig_7](images/img_007.png)



---



![fig_8](images/img_008.png)

## SESSION OBJECTIVES

At this point in the course, we are about t

In [ ]:
from PIL import Image
import numpy as np
from skimage.metrics import structural_similarity as ssim

def compare_images_ssim(path1, path2):
    img1 = np.array(Image.open(path1).convert("L")) / 255.0
    img2 = np.array(Image.open(path2).convert("L")) / 255.0

    # Resize if needed
    if img1.shape != img2.shape:
        from skimage.transform import resize
        img2 = resize(img2, img1.shape, anti_aliasing=True)

    score, _ = ssim(img1, img2, full=True, data_range=1.0)
    return score

SESSION6_DIR = OUTPUT_DIR / Path("markdown") / Path("Session 06 - Regular Expressions") / Path("images")
score = compare_images_ssim(SESSION6_DIR / Path("img_002.png"), SESSION6_DIR / Path("img_005.png"))
print(score)

0.9621646190945281
True
